In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler 


Up untill now I am just downloading the dataset, but in this dataset the index is multi-index so wi first flattened it

In [2]:
# now because csv dont parse the date time so this might be an issue for me, as the data will be trained in the future project based on 
# rolling window approach so it is really crucial to make sure that the data has the datetime sorted 

FinalStocks = pd.read_csv('data/raw_stocks.csv', parse_dates=["Date"])
# this is the first step to clean the data cleaning and making sure that the date column is of type datetime only 

In [8]:
FinalStocks[FinalStocks["Ticker"] == 'CBA.AX'].head()

,Date,Close,High,Low,Open,Volume,Ticker
4109,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX
4110,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX
4111,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX
4112,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX
4113,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX


In [9]:
print(FinalStocks['Ticker'].unique())

['BHP.AX' 'CBA.AX' 'CSL.AX' 'NAB.AX' 'WBC.AX' 'ANZ.AX' 'WES.AX' 'MQG.AX'
 'WOW.AX' 'TLS.AX']


In [10]:
CBA_Stock = FinalStocks[FinalStocks['Ticker'] == 'CBA.AX'].copy()

In [11]:
CBA_Stock = CBA_Stock.sort_values('Date').reset_index(drop=True)

In [12]:
print("Shape:", CBA_Stock.shape)

print("\nDate Range:")
print(CBA_Stock['Date'].min(), "→", CBA_Stock['Date'].max())

print("\nCheck ordering:")
print(CBA_Stock['Date'].is_monotonic_increasing)

print("\nMissing values:")
print(CBA_Stock.isnull().sum())

Shape: (4109, 7)

Date Range:
2010-01-04 00:00:00 → 2026-04-02 00:00:00

Check ordering:
True

Missing values:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Ticker    0
dtype: int64


In [13]:
CBA_Stock['Return'] = CBA_Stock['Close'].pct_change()

CBA_Stock['LogReturn'] = np.log(CBA_Stock['Close'] / CBA_Stock['Close'].shift(1))

In [14]:
CBA_Stock.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904


In [15]:
CBA_Stock['SMA_5'] = CBA_Stock['Close'].rolling(5).mean()
CBA_Stock['SMA_10'] = CBA_Stock['Close'].rolling(10).mean()

In [17]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,SMA_10
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,NaN
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,NaN
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,NaN
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,NaN
5,2010-01-11,25.390263,25.484516,25.246637,25.269078,2751882,CBA.AX,0.007300,0.007274,25.120963,NaN
6,2010-01-12,25.390263,25.592236,25.291520,25.336403,3539936,CBA.AX,0.000000,0.000000,25.199060,NaN
7,2010-01-13,25.138912,25.354350,25.071589,25.255607,2761569,CBA.AX,-0.009899,-0.009949,25.201753,NaN
8,2010-01-14,25.489004,25.623652,25.269077,25.313959,3020355,CBA.AX,0.013926,0.013830,25.322937,NaN
9,2010-01-15,26.076965,26.076965,25.224190,25.493487,4801453,CBA.AX,0.023067,0.022805,25.497081,25.232721


In [18]:
CBA_Stock['Return_std_5'] = CBA_Stock['Return'].rolling(5).std()
CBA_Stock['Return_std_10'] = CBA_Stock['Return'].rolling(10).std()

In [19]:
CBA_Stock['Return_lag1'] = CBA_Stock['Return'].shift(1)
CBA_Stock['Return_lag2'] = CBA_Stock['Return'].shift(2)
CBA_Stock['Return_lag3'] = CBA_Stock['Return'].shift(3)
CBA_Stock['Return_lag5'] = CBA_Stock['Return'].shift(5)

In [20]:
CBA_Stock['Momentum_5'] = CBA_Stock['Return'].rolling(5).sum()
CBA_Stock['Momentum_10'] = CBA_Stock['Return'].rolling(10).sum()

In [21]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,SMA_10,Return_std_5,Return_std_10,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,NaN,NaN,NaN,0.015126,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,NaN,NaN,NaN,0.005027,0.015126,NaN,NaN,NaN,NaN
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,NaN,NaN,NaN,-0.009646,0.005027,0.015126,NaN,NaN,NaN
5,2010-01-11,25.390263,25.484516,25.246637,25.269078,2751882,CBA.AX,0.007300,0.007274,25.120963,NaN,0.009739,NaN,0.012987,-0.009646,0.005027,NaN,0.030795,NaN
6,2010-01-12,25.390263,25.592236,25.291520,25.336403,3539936,CBA.AX,0.000000,0.000000,25.199060,NaN,0.008532,NaN,0.007300,0.012987,-0.009646,0.015126,0.015668,NaN
7,2010-01-13,25.138912,25.354350,25.071589,25.255607,2761569,CBA.AX,-0.009899,-0.009949,25.201753,NaN,0.010160,NaN,0.000000,0.007300,0.012987,0.005027,0.000742,NaN
8,2010-01-14,25.489004,25.623652,25.269077,25.313959,3020355,CBA.AX,0.013926,0.013830,25.322937,NaN,0.009946,NaN,-0.009899,0.000000,0.007300,-0.009646,0.024314,NaN
9,2010-01-15,26.076965,26.076965,25.224190,25.493487,4801453,CBA.AX,0.023067,0.022805,25.497081,25.232721,0.012656,NaN,0.013926,-0.009899,0.000000,0.012987,0.034395,NaN


In [22]:
CBA_Stock['Close_SMA5_diff'] = CBA_Stock['Close'] - CBA_Stock['SMA_5']
CBA_Stock['Close_SMA10_diff'] = CBA_Stock['Close'] - CBA_Stock['SMA_10']

In [24]:
CBA_Stock['TargetReturn'] = CBA_Stock['Return'].shift(-1)
CBA_Stock['TargetTrend'] = (CBA_Stock['TargetReturn'] > 0).astype(int)

In [25]:
CBA_Stock.head()

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-04,24.627253,24.672137,24.438745,24.492605,992762,CBA.AX,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015126,1
1,2010-01-05,24.999775,25.044659,24.708037,24.708037,3163161,CBA.AX,0.015126,0.015013,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005027,1
2,2010-01-06,25.125446,25.224189,25.040169,25.138911,3210425,CBA.AX,0.005027,0.005014,NaN,...,0.015126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009646,0
3,2010-01-07,24.883083,25.345379,24.775365,25.134428,3089618,CBA.AX,-0.009646,-0.009693,NaN,...,0.005027,0.015126,NaN,NaN,NaN,NaN,NaN,NaN,0.012987,1
4,2010-01-08,25.206245,25.269081,24.986320,25.008760,3159303,CBA.AX,0.012987,0.012904,24.968361,...,-0.009646,0.005027,0.015126,NaN,NaN,NaN,0.237885,NaN,0.007300,1


In [26]:
CBA_Stock = CBA_Stock.dropna().reset_index(drop=True)

In [31]:
print(CBA_Stock['TargetTrend'].value_counts(normalize=True))

TargetTrend
1    0.526354
0    0.473646
Name: proportion, dtype: float64


In [27]:
CBA_Stock.head(10)

,Date,Close,High,Low,Open,Volume,Ticker,Return,LogReturn,SMA_5,...,Return_lag1,Return_lag2,Return_lag3,Return_lag5,Momentum_5,Momentum_10,Close_SMA5_diff,Close_SMA10_diff,TargetReturn,TargetTrend
0,2010-01-18,26.054525,26.323823,25.816647,26.054525,6117434,CBA.AX,-0.000861,-0.000861,25.629934,...,0.023067,0.013926,-0.009899,0.007300,0.026234,0.057028,0.424591,0.679077,-0.024289,0
1,2010-01-19,25.421679,26.027597,25.381284,26.014132,5840218,CBA.AX,-0.024289,-0.024589,25.636217,...,-0.000861,0.023067,0.013926,0.000000,0.001944,0.017613,-0.214539,0.004040,-0.001412,0
2,2010-01-20,25.385778,25.722399,25.278059,25.717912,3761697,CBA.AX,-0.001412,-0.001413,25.685590,...,-0.024289,-0.000861,0.023067,-0.009899,0.010432,0.011174,-0.299812,-0.057893,0.000000,0
3,2010-01-21,25.385778,25.699959,25.273571,25.381289,5142694,CBA.AX,0.000000,0.000000,25.664945,...,-0.001412,-0.024289,-0.000861,0.013926,-0.003495,0.020820,-0.279167,-0.108163,-0.014321,0
4,2010-01-22,25.022221,25.085056,24.690087,24.910013,5067217,CBA.AX,-0.014321,-0.014425,25.453996,...,0.000000,-0.001412,-0.024289,0.023067,-0.040883,-0.006489,-0.431776,-0.453318,-0.010762,0
5,2010-01-25,24.752924,24.842689,24.577881,24.685599,3028846,CBA.AX,-0.010762,-0.010821,25.193676,...,-0.014321,0.000000,-0.001412,-0.000861,-0.050785,-0.024552,-0.440752,-0.658881,-0.009429,0
6,2010-01-27,24.519535,24.645208,24.326538,24.604813,4035586,CBA.AX,-0.009429,-0.009473,25.013247,...,-0.010762,-0.014321,0.000000,-0.024289,-0.035925,-0.033980,-0.493712,-0.805197,0.008603,1
7,2010-01-28,24.730480,24.775363,24.550948,24.721503,5277686,CBA.AX,0.008603,0.008566,24.882188,...,-0.009429,-0.010762,-0.014321,-0.001412,-0.025909,-0.015478,-0.151707,-0.553409,-0.033938,0
8,2010-01-29,23.891169,24.434252,23.787938,24.371415,9940798,CBA.AX,-0.033938,-0.034528,24.583266,...,0.008603,-0.009429,-0.010762,0.000000,-0.059848,-0.063342,-0.692097,-1.232937,-0.004697,0
9,2010-02-01,23.778961,24.120072,23.778961,23.877704,4273994,CBA.AX,-0.004697,-0.004708,24.334614,...,-0.033938,0.008603,-0.009429,-0.014321,-0.050223,-0.091106,-0.555653,-1.115344,0.010759,1


In [32]:
CBA_Stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              4098 non-null   datetime64[ns]
 1   Close             4098 non-null   float64       
 2   High              4098 non-null   float64       
 3   Low               4098 non-null   float64       
 4   Open              4098 non-null   float64       
 5   Volume            4098 non-null   int64         
 6   Ticker            4098 non-null   object        
 7   Return            4098 non-null   float64       
 8   LogReturn         4098 non-null   float64       
 9   SMA_5             4098 non-null   float64       
 10  SMA_10            4098 non-null   float64       
 11  Return_std_5      4098 non-null   float64       
 12  Return_std_10     4098 non-null   float64       
 13  Return_lag1       4098 non-null   float64       
 14  Return_lag2       4098 n

In [33]:
feature_columns = [
    'Return',
    'Volume',

    'Return_lag1',
    'Return_lag2',
    'Return_lag3',
    'Return_lag5',

    'Momentum_5',
    'Momentum_10',

    'Return_std_5',
    'Return_std_10',

    'Close_SMA5_diff',
    'Close_SMA10_diff'
]